# Audit, analisis LDA tervalidasi, dan eksperimen wisata — v2

**Malik Hariyanto · Asistensi Hibah Penelitian**

Dataset dimuat otomatis; jalankan sel berurutan. Versi ini memisahkan train/validation/test per destinasi, membandingkan jumlah topik dan seed, menampilkan contoh ulasan, serta memilih checkpoint BiLSTM berdasarkan validation loss.

**Landasan setiap keputusan dan parameter:** [METODOLOGI_DAN_PAPER.md](https://github.com/MalikHariyanto/crispy_recom_machine_system/blob/feat/streamlit-wisata/docs/METODOLOGI_DAN_PAPER.md). ID R01–R14 di bawah merujuk daftar tersebut. Nilai hiperparameter adalah pengaturan eksperimen, bukan angka yang diklaim diwajibkan paper.

Dataset terbaru sepenuhnya destinasi wisata Madura sesuai konfirmasi pemilik data: 2.058 ulasan, 27 destinasi. Terdapat dua atribut destinasi serta tanggal relatif. Topik belum merupakan aspek/sentimen tervalidasi. Notebook ini tidak mengimplementasikan fusion BGE-E5 pada sertifikat HKI.

## 00 Konfigurasi dan instalasi

Biarkan `DATA_PATH = None`: tidak perlu upload dataset. `RUN_DEEP=True` melatih dua BiLSTM; GPU disarankan. `RUN_DEEP=False` tetap menjalankan seluruh analisis LDA, baseline, grafik, dan ekspor.

Pencarian default: 6 jumlah topik × 3 seed = 18 LDA. Semua parameter terlihat di sini; batas komputasi dan alasan pengaturan dijelaskan pada dokumen metodologi. R01–R05, R07–R13.

In [ ]:
from datetime import datetime
from pathlib import Path

DATA_PATH = None
OUTPUT_DIR = Path("hasil_wisata_madura") / datetime.now().strftime("%Y%m%d-%H%M%S-%f")
JUDGMENTS_PATH = None
SEED = 42
VALIDATION_SEED = 142
TEST_FRACTION = 0.20
VALIDATION_FRACTION = 0.20  # Dari sisa train; tidak mengubah test awal.
TOPIC_COUNTS = (2, 4, 6, 8, 10, 12)
LDA_SEEDS = (42, 43, 44)
LDA_MAX_ITER = 20
LDA_REVIEWS_PER_DESTINATION = 30
RUN_DEEP = True
RUN_BGE = False
MAX_BILSTM_EPOCHS = 15
EARLY_STOPPING_PATIENCE = 3

In [ ]:
import subprocess
import sys
import os
import hashlib

cwd = Path.cwd()
checkout = cwd / "wisata_madura_colab"
candidates = (cwd, cwd.parent, checkout)
repo_root = next((p for p in candidates if
    (p / "src" / "wisata_madura" / "visualization.py").is_file()
    and (p / "data" / "manifest.json").is_file()), None)
if repo_root is None:
    if checkout.exists():
        raise FileExistsError(f"{checkout} belum memuat kode terbaru. Ganti nama folder lama, lalu ulangi sel ini.")
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "feat/streamlit-wisata",
                    "https://github.com/MalikHariyanto/crispy_recom_machine_system.git", str(checkout)], check=True)
    repo_root = checkout
requirements = ["-r", str(repo_root / "requirements-wisata.txt")]
if RUN_DEEP:
    requirements += ["gensim>=4.4,<5", "torch>=2.2,<3"]
if RUN_BGE:
    requirements += ["sentence-transformers>=3,<6"]
subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", *requirements], check=True)
# Kernel yang sudah mengimpor modul versi lama harus direstart, agar hasil tidak mencampur versi.
if any(name.startswith("wisata_madura") for name in sys.modules):
    raise RuntimeError("Modul sudah pernah diimpor. Restart runtime/kernel dan jalankan dari awal.")
sys.path.insert(0, str(repo_root / "src"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str((OUTPUT_DIR / ".matplotlib").resolve()))
source_files = list((repo_root / "src" / "wisata_madura").glob("*.py"))
source_hashes = {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in source_files}
source_commit = subprocess.check_output(["git", "-C", str(repo_root), "rev-parse", "HEAD"], text=True).strip() if (repo_root / ".git").exists() else "zip_snapshot; see source hashes"
print("Source:", source_commit)

## 01 Muat dataset Madura otomatis dan periksa skema

Dataset terbaru: **dataset_madura_bersih.xlsx**, sheet **Sheet1**, 2.058 ulasan dan 27 destinasi wisata Madura. Dataset dimuat otomatis dari repository; tidak perlu unggah file atau CSV wilayah. `DATA_PATH` hanya untuk eksperimen dengan workbook alternatif.

Checksum workbook dan paket model diperiksa. Kolom rating asli dibaca untuk eksplorasi distribusi, sedangkan model tetap memakai teks `text_preprocessed` yang dinormalisasi ringan tanpa membuang negasi. Cakupan seluruh dataset adalah Madura sesuai konfirmasi pemilik data, bukan hasil geocoding otomatis.


In [ ]:
import json
from importlib.metadata import version

import pandas as pd
from IPython.display import display

from wisata_madura.data import (
    audit_reviews, label_distribution, load_bundled_reviews, load_reviews, split_reviews,
)
from wisata_madura.modeling import evaluate_classifiers, fit_topics, run_baselines, run_lda_only

if DATA_PATH is None:
    bundled, data_manifest = load_bundled_reviews(repo_root / "data")
    workbook_path = repo_root / "data" / "source" / data_manifest["source_workbook"]
    if hashlib.sha256(workbook_path.read_bytes()).hexdigest() != data_manifest["source_workbook_sha256"]:
        raise ValueError("Checksum workbook sumber tidak cocok dengan manifest.")
    reviews = load_reviews(workbook_path)
    columns = ["id_destinasi", "destinasi", "wisata", "rekomendasi_cuaca", "teks_model"]
    pd.testing.assert_frame_equal(reviews[columns], bundled[columns])
    print("Dataset Madura terbaru dimuat otomatis beserta rating asli.")
else:
    data_manifest = None
    workbook_path = Path(DATA_PATH)
    reviews = load_reviews(workbook_path)
    print(f"Dataset alternatif dimuat dari {DATA_PATH}.")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
geography = {"scope": "madura_complete_dataset", "basis": "owner_confirmed",
             "source_destinations": int(reviews.id_destinasi.nunique())} if DATA_PATH is None else {
             "scope": "user_supplied_alternative", "source_destinations": int(reviews.id_destinasi.nunique())}
print(f"{len(reviews):,} ulasan dari {reviews['id_destinasi'].nunique()} destinasi")
print(list(reviews.columns))


## 02 Audit kualitas dan distribusi label

**Keluaran:** jumlah teks kosong, duplikat persis, tanggal relatif, indikator karakter rusak, dan distribusi atribut. Indikator karakter rusak membantu memilih sampel untuk diperiksa. Pada paket bawaan, hitungan tanggal relatif dan karakter rusak menggunakan penanda yang dihitung dari workbook sumber.

In [ ]:
audit = audit_reviews(reviews, workbook_path)
audit["geography"] = geography
if data_manifest is not None:
    audit["source_workbook_sha256"] = data_manifest["source_workbook_sha256"]
distribution = label_distribution(reviews)
display(pd.Series({key: audit[key] for key in (
    "rows", "destinations", "exact_duplicate_rows", "empty_normalized_text_rows",
    "relative_date_rows", "potential_encoding_error_rows",
)}))
display(distribution)
distribution.to_csv(OUTPUT_DIR / "distribusi_label.csv", index=False)


## 02B Visualisasi rating, cuaca, jenis wisata, dan panjang ulasan

- **Diagram batang rating 1–5:** jumlah ulasan untuk setiap bintang.
- **Diagram batang cuaca:** jumlah ulasan per label cuaca (kategori, bukan data numerik kontinu).
- **Diagram lingkaran/donut + persentase:** cuaca dan jenis wisata per **destinasi unik**; destinasi ramai tidak dihitung berkali-kali.
- **Histogram panjang teks:** jumlah kata per ulasan setelah preprocessing.

Seluruh gambar PNG dan tabel CSV disimpan ke folder run. Label cuaca menggambarkan atribut dataset, bukan prakiraan cuaca harian.


In [ ]:
import matplotlib.pyplot as plt
from wisata_madura.visualization import plot_dataset_overview

eda_figures, eda_tables = plot_dataset_overview(reviews, OUTPUT_DIR)
for figure in eda_figures:
    display(figure)
    plt.close(figure)
for name in ("rating", "cuaca_ulasan", "cuaca_destinasi", "jenis_destinasi"):
    print(name)
    display(eda_tables[name].round(2))
print("Rating kosong/tidak valid:", eda_tables["rating_tidak_valid"])


## 03 Pisahkan train, validation, dan test

R06: evaluasi harus memperhitungkan struktur kelompok data. Test awal tetap sekitar 20% destinasi; validation diambil sekitar 20% dari destinasi train sisanya. Semua model dilatih pada subset train yang sama. ID dan teks identik diperiksa agar tidak berbagi antarpartisi. Rasio ulasan tidak persis 64:16:20 karena ukuran destinasi berbeda.

Validation dipakai memilih jumlah topik dan checkpoint; test hanya untuk evaluasi akhir. Kelas langka bisa tidak muncul pada validation—lihat audit. Hasil v2 tidak langsung sebanding dengan angka v1 yang memakai train lebih besar.
Dengan dataset terbaru dan seed default, pembagian destinasi menghasilkan 16 train, 5 validation, dan 6 test. Ulasan setelah pembersihan: 342 train, 108 validation, 1.446 test (1.896 total, dari 2.058 awal). Banyak ulasan berada pada beberapa destinasi besar, sehingga proporsi ulasan dapat sangat timpang walaupun proporsi destinasi mendekati target. Angka grafik selalu dihitung ulang, bukan konstanta tersebut.


In [ ]:
from wisata_madura.data import validation_split
outer_split = split_reviews(reviews, test_fraction=TEST_FRACTION, seed=SEED)
split = validation_split(outer_split, fraction=VALIDATION_FRACTION, seed=VALIDATION_SEED)
audit.update(split.audit)
display(pd.DataFrame({"ulasan": split.audit["partition_reviews"],
                      "destinasi": split.audit["partition_destinations"]}))
print("Kelas tidak hadir pada validation:", split.audit["validation_split"]["classes_absent_from_test"])
for name, frame in (("train", split.train), ("validation", split.validation), ("test", split.test)):
    frame[["id_destinasi", "wisata", "rekomendasi_cuaca"]].drop_duplicates().assign(partisi=name).to_csv(
        OUTPUT_DIR / f"id_partisi_{name}.csv", index=False)
# Diagram aktual setelah pembersihan dan pemisahan per destinasi.
from wisata_madura.visualization import plot_partition_overview
partition_figure, partition_table = plot_partition_overview(split, OUTPUT_DIR)
display(partition_table.round(2))
display(partition_figure)
plt.close(partition_figure)
print("Rasio di atas dihitung dari data aktual, bukan dipaksakan menjadi 64:16:20.")
print("Duplikat/teks kosong/teks identik antarpartisi dapat membuat jumlah ulasan model lebih kecil dari dataset awal.")


## 04 Latih model pembanding

R14: TF-IDF + Linear SVC dibandingkan dengan kelas mayoritas. Fit hanya train; vocabulary tidak mempelajari validation/test. Partisi train ini sama dengan yang dipakai LDA dan BiLSTM.

In [ ]:
predictions = run_baselines(split, seed=SEED)
print("Model pembanding:", list(predictions))

## 05A Cari jumlah topik, ukur coherence, perplexity, dan kestabilan

R01–R05: LDA memakai count kata (bukan embedding). Kosakata dan model hanya fit pada sampel train, maksimal 30 ulasan per destinasi. Stopword Indonesia/Inggris eksplisit disimpan di audit; negasi dipertahankan. Ini berlaku untuk LDA, bukan menghapus kata dari masukan FastText/TF-IDF.

Bandingkan K dan seed pada **validation**. Coherence di sini adalah **NPMI per dokumen**, bukan c_v atau NPMI dengan sliding window; pasangan tidak muncul bersama diberi -1. Pilih K dari rerata validation NPMI; jika seri, stability lalu perplexity. Gunakan seed utama yang sudah ditentukan. Stabilitas berupa top-word Jaccard dengan pencocokan topik optimal; ini adaptasi diagnosis, bukan reproduksi lengkap Greene dkk.

Coherence tinggi relatif belum membuktikan aspek bermakna. NPMI negatif/topik tidak stabil perlu dibaca bersama contoh ulasan. Model tidak dilatih ulang pada validation setelah pencarian.

In [ ]:
from wisata_madura.topics import TopicConfig, analyze_topics
from IPython.display import Image
from dataclasses import asdict

topic_config = TopicConfig(topic_counts=TOPIC_COUNTS, seeds=LDA_SEEDS,
                          max_iter=LDA_MAX_ITER, per_destination=LDA_REVIEWS_PER_DESTINATION)
topics, topic_tables, topic_audit = analyze_topics(split, OUTPUT_DIR, topic_config)
audit["lda"] = topic_audit
print("K terpilih:", topic_audit["selected_topics"])
display(topic_tables["pemilihan_topik"].round(4))
display(topic_tables["kualitas_topik"].round(4))
for filename in ("pemilihan_topik.png", "kata_utama_topik.png", "heatmap_topik_cuaca.png"):
    display(Image(filename=str(OUTPUT_DIR / filename)))

## 05B Baca contoh ulasan dan evaluasi fitur LDA

R03–R05: periksa kata dan ulasan yang topik dominannya cocok. Topik yang tidak dominan pada dokumen mana pun dapat tidak punya contoh. Nama aspek tidak diisi otomatis; gunakan `template_review_topik.csv` setelah membaca contoh.

Heatmap cuaca hanya deskriptif pada train sample dan ditimbang per destinasi. Perbedaan bobot bukan bukti perubahan antar-musim kunjungan. LDA + regresi logistik adalah klasifikasi atribut dari fitur topik; ini berbeda dari menilai coherence topik.

In [ ]:
pd.set_option("display.max_colwidth", 160)
display(topic_tables["contoh_ulasan_topik"])
print("Review manual: baca contoh, isi nama/catatan pada template_review_topik.csv.")
predictions.update(run_lda_only(split, topics, seed=SEED))


## 06 FastText dari train saja

R07: representasi subword FastText dipelajari dari train saja. Embedding dibekukan saat BiLSTM dilatih. Kata validation/test baru mendapat vektor subword dari model tersebut, tanpa melatih ulang pada validation/test. Panjang urutan, dimensi, dan epoch adalah pengaturan eksperimen; lihat metodologi.

In [ ]:
if RUN_DEEP:
    from wisata_madura.neural import DeepConfig, fit_fasttext, run_bilstm_variants, plot_training_history
    deep_config = DeepConfig(seed=SEED, bilstm_epochs=MAX_BILSTM_EPOCHS,
                             patience=EARLY_STOPPING_PATIENCE)
    encoded = fit_fasttext(split, deep_config, output_dir=OUTPUT_DIR / "models")
    audit["deep_config"] = asdict(deep_config)
    print("Train:", encoded.train_tokens.shape, "Validation:", encoded.validation_tokens.shape)
    print("Embedding:", encoded.embeddings.shape)
else:
    print("FastText/BiLSTM tidak dijalankan pada run ini.")

## 07 BiLSTM, validation loss, early stopping, dan checkpoint

R08–R12: dua varian mempertahankan arsitektur urutan, split, optimizer, dan anggaran epoch yang sama. Satu kepala softmax memprediksi setiap atribut (bukan sentimen multi-label).

Validation loss memakai bobot kelas dari train; tidak menggunakan test. Hentikan setelah beberapa epoch tanpa perbaikan, pulihkan checkpoint terbaik, kemudian prediksi test. Grafik memperlihatkan train/validation loss. R13: satu seed deep-learning belum mengukur variasi antar-run; tiga seed pada pencarian LDA tidak berarti BiLSTM sudah diuji tiga seed.

In [ ]:
if RUN_DEEP:
    deep_predictions, training_history = run_bilstm_variants(
        split, topics, encoded, deep_config, output_dir=OUTPUT_DIR / "models")
    predictions.update(deep_predictions)
    training_history.to_csv(OUTPUT_DIR / "riwayat_bilstm.csv", index=False)
    plot_training_history(training_history, OUTPUT_DIR)
    display(training_history)
    display(pd.read_csv(OUTPUT_DIR / "models" / "checkpoint_bilstm.csv"))
    display(Image(filename=str(OUTPUT_DIR / "kurva_bilstm.png")))
else:
    print("Evaluasi berikut hanya untuk model yang benar-benar dijalankan.")

## 08 Evaluasi akhir pada test

R06, R13–R14: tampilkan accuracy, micro-F1, macro-F1, Hamming loss, serta hasil per kelas. Prediksi destinasi adalah suara terbanyak ulasan. Perbandingan fitur tidak menjamin LDA meningkatkan hasil.

Karena setiap atribut satu label per contoh, micro-F1 setara accuracy dan Hamming loss = 1 - accuracy. Jangan menganggap ketiganya bukti independen. Test tetap satu partisi 43 destinasi pada paket lengkap; kesimpulan stabil memerlukan evaluasi berulang terpisah.

In [ ]:
metrics, per_class = evaluate_classifiers(split.test, predictions)
display(metrics.loc[metrics["unit"].eq("destinasi")].round(4))
metrics.to_csv(OUTPUT_DIR / "metrik_klasifikasi.csv", index=False)
per_class.to_csv(OUTPUT_DIR / "metrik_per_kelas.csv", index=False)
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10, 4), layout="constrained")
metrics.loc[metrics.unit.eq("destinasi")].pivot(index="model", columns="target", values="macro_f1").plot.bar(ax=ax)
ax.set(ylabel="Macro-F1", title="Test destinasi — semua model pada split yang sama", ylim=(0, 1))
fig.savefig(OUTPUT_DIR / "perbandingan_model.png", dpi=160)
plt.close(fig)
display(Image(filename=str(OUTPUT_DIR / "perbandingan_model.png")))

## 09 Rekomendasikan destinasi dengan BGE

**Masukan:** preferensi, konteks Kemarau/Penghujan jika diketahui, dan ID kunjungan opsional. BGE-M3 memberi skor kemiripan semantik pada deskripsi destinasi. Atribut cuaca pada workbook menyaring kandidat sesuai pilihan pengguna. Modul memakai katalog yang tersedia sebagai tahap pemeringkatan. Keluaran klasifikasi dapat dipertimbangkan untuk katalog baru setelah modelnya tervalidasi.

Ubah preferensi dan musim di sel berikut. Peringkat contoh tidak digunakan sebagai penilaian relevansi. Isi `JUDGMENTS_PATH` dengan CSV relevansi independen berkolom `query_id,preferensi,musim,id_destinasi` untuk menghitung Precision@K dan Recall@K.

In [ ]:
from wisata_madura.recommendation import BGERecommender, build_catalog, evaluate_recommendations

PREFERENSI = "Wisata alam dengan pemandangan yang cocok untuk keluarga"
MUSIM = "Penghujan"
RIWAYAT_IDS = []
K = 5

catalog = build_catalog(reviews)
print(f"{len(catalog)} destinasi dalam katalog dengan cakupan {geography['scope']}.")

if RUN_BGE:
    recommender = BGERecommender.from_pretrained(catalog)
    ranking = recommender.recommend(PREFERENSI, MUSIM, k=K, riwayat_ids=RIWAYAT_IDS)
    display(ranking)
    ranking.assign(cakupan=geography["scope"]).to_csv(OUTPUT_DIR / "contoh_rekomendasi.csv", index=False)
    if JUDGMENTS_PATH is not None:
        judgments = pd.read_csv(JUDGMENTS_PATH, dtype={"id_destinasi": str})
        ranking_metrics = evaluate_recommendations(recommender, judgments, k=K)
        display(ranking_metrics)
        display(ranking_metrics[["precision_at_k", "recall_at_k"]].mean())
        ranking_metrics.to_csv(OUTPUT_DIR / "metrik_rekomendasi.csv", index=False)
    else:
        print("Precision@K dan Recall@K memerlukan CSV penilaian relevansi independen.")
else:
    print("Katalog siap. Aktifkan RUN_BGE di bagian 00 untuk memeringkat destinasi.")

## 10 Simpan provenance, parameter, hasil, dan model

Seed, versi paket, commit/hash modul, konfigurasi LDA, split, serta checkpoint disimpan. Satu run satu folder. Berkas model hanya dimuat jika berasal dari run/sumber yang dipercaya.

Artefak baru: tabel pencarian, coherence per topik, kestabilan, bobot kata, contoh ulasan, template review, grafik, model/vectorizer LDA, dan (jika deep aktif) model FastText serta checkpoint BiLSTM.

Dataset aktif sepenuhnya Madura sesuai konfirmasi pemilik data. Asal label cuaca, anotasi sentimen, dan relevansi rekomendasi tetap perlu didokumentasikan. R01–R14 serta batas adaptasi implementasi ada di `docs/METODOLOGI_DAN_PAPER.md`.

In [ ]:
packages = ["numpy", "pandas", "scikit-learn", "openpyxl", "matplotlib"]
if RUN_DEEP:
    packages += ["gensim", "torch"]
if RUN_BGE:
    packages += ["sentence-transformers"]

audit.update({
    "python": sys.version.split()[0],
    "pipeline_version": "2.0",
    "source_commit": source_commit,
    "source_module_sha256": source_hashes,
    "packages": {package: version(package) for package in packages},
    "run_deep": RUN_DEEP,
    "run_bge": RUN_BGE,
})
(OUTPUT_DIR / "audit_dataset.json").write_text(
    json.dumps(audit, indent=2, ensure_ascii=False), encoding="utf-8"
)
print("Keluaran:", OUTPUT_DIR.resolve())
print("\n".join(f.name for f in sorted(OUTPUT_DIR.iterdir()) if f.is_file()))